<!--
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
-->

# GÜN 050: Virtual Function ve Polymorphism (Sanal Fonksiyonlar, Dinamik Çokbiçimlilik ve vtable Mimarisi)

[![License: All Rights Reserved](https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square)](https://github.com/seydivakkas)
[![C++ Standard](https://img.shields.io/badge/C%2B%2B-20-blue?style=flat-square&logo=c%2B%2B)](https://en.cppreference.com/w/cpp/20)
[![Domain](https://img.shields.io/badge/Domain-Air%20Defense%20%26%20Command%20Control-darkgreen?style=flat-square)]()
[![Safety](https://img.shields.io/badge/Safety-MISRA%20C%2B%2B%3A2023%20%7C%20DO--178C%20DAL--A-orange?style=flat-square)]()

---

### 1. 5N1K Kurumsal Analiz Çerçevesi

| Soru | Sistem Mühendisliği Yanıtı |
| :--- | :--- |
| **Ne? (What)** | **Virtual Function (Sanal Fonksiyon)**, çalışma zamanında nesnenin gerçek dinamik tipine göre çağrılan üye fonksiyondur; **Polymorphism (Çokbiçimlilik)** ise tek bir arayüz arkasında farklı türemiş sınıfların kendi özel davranışlarını sergileme yeteneğidir. |
| **Neden? (Why)** | Savunma sistemlerinde komuta kontrol (C2) yazılımları farklı füzeleri, radarları veya namlulu silahları tek bir standart arayüzden yönetmek zorundadır. Aksi halde her yeni silah için tüm komuta yazılımı if-else bloklarıyla baştan yazılmak zorunda kalır (Spaghetti Architecture). |
| **Nasıl? (How)** | C++ derleyicisi sınıflar için bir sanal fonksiyon tablosu (**`vtable`**) ve her nesnenin içine 8 baytlık bir sanal tablo göstericisi (**`__vptr`**) yerleştirerek dinamik bağlamayı (Dynamic Dispatch) dolaylı atlama ile yürütür. |
| **Nerede? (Where)** | ASELSAN HİSAR/SİPER hava savunma komuta kontrol yazılımlarında, HAVELSAN ADVENT savaş yönetim sisteminde ve TUSAŞ uçuş kontrol yüzey aktüatör yöneticilerinde. |
| **Ne Zaman? (When)** | Farklı donanım veya alt sistemlerin ortak bir soyut arayüzü paylaştığı ve hangi somut sistemin çalışacağının çalışma zamanında (runtime) belirlendiği senaryolarda. |
| **Kim? (Who)** | Savaş yönetim sistemi mimarları, gerçek zamanlı C++20 gömülü sistem mühendisleri ve aviyonik yazılım uzmanları. |

---

### 2. Pratik Hayat ve Savunma Sanayii Analojisi

> **Hava Savunma Komuta Kontrol Karargahı ve Önleme Butonu:**
> - Bir düşman balistik füzesi veya seyir füzesi radara girdiğinde, Komuta Kontrol Konsolu'ndaki nöbetçi subayın önünde tek bir kırmızı düğme vardır: **"HEDEFİ İMHA ET (`engageTarget()`)"**.
> - Konsol yazılımı (Taban Sınıf Arayüzü: `IAirDefenseInterceptor`) hedefin menziline ve irtifasına bakar:
>   - Hedef 4 km'de alçaktan uçuyorsa düğmeye basıldığında **KORKUT 35mm Parçacıklı Namlulu Silah** ateş açar.
>   - Hedef 15 km'deyse **HİSAR-A+ Füzesi** fırlar.
>   - Hedef 60 km'deyse **HİSAR-O+ Füzesi** dikey atımla ateşlenir.
>   - Hedef 100+ km'de yüksek irtifadaysa **SİPER Blok-1 Uzun Menzilli Bölge Savunma Füzesi** devreye girer.
> - Subayın önündeki düğme aynı düğmedir (`virtual void engage()`); ancak arkada çalışan fiziksel mekanizma hedefe göre çokbiçimli (polymorphic) olarak değişir!


## 1. Sanal Fonksiyonların Makine Modeli: `vtable` ve `__vptr`

### 1.1 Derleyicinin Perde Arkası

C++'ta bir sınıfta en az bir adet `virtual` fonksiyon tanımlandığında derleyici şu iki kritik yapıyı inşa eder:

1. **`vtable` (Virtual Method Table):**
   - Programın salt-okunur veri segmentinde (`.rodata`) yaşayan bir fonksiyon gösterici dizisidir (`void (*vtable[])()`).
   - Her polimorfik sınıf için RAM'de **yalnızca tek bir kopya** bulunur.
   - Sınıfın tüm sanal metotlarının başlangıç bellek adreslerini sırasıyla tutar.

2. **`__vptr` (Virtual Table Pointer):**
   - Sınıftan oluşturulan her bir nesnenin en başına derleyici tarafından gizlice eklenen **64-bitlik (8 bayt)** bir adrestir (`sizeof(vptr) = 8`).
   - Nesne inşa edildiğinde (Constructor çalıştığında), nesnenin `__vptr`'ı doğrudan o sınıfın `vtable` adresine mühürlenir.

```text
[ STACK / HEAP NESNESİ ]                     [ .rodata SEGMENTİ (vtable) ]
┌─────────────────────────┐                  ┌───────────────────────────────┐
│ __vptr (Offset 0x0, 8B) ├─────────────────►│ 0: &Derived::engageTarget     │
├─────────────────────────┤                  ├───────────────────────────────┤
│ m_launcher_id (4B)      │                  │ 1: &Derived::~Derived (Dtor)  │
├─────────────────────────┤                  └───────────────────────────────┘
│ Padding (4B)            │
└─────────────────────────┘
```

---

### 1.2 Dinamik Bağlamanın (Dynamic Dispatch) Assembly İspatı

Aşağıdaki çağrının derlenmiş x86_64 makine kodunu inceleyelim:

```cpp
IAirDefenseInterceptor* interceptor = getSystem();
interceptor->engageTarget(target);
```

Derleyicinin ürettiği Assembly adımları:
```assembly
mov rax, [rdi]        ; 1. ADIM: rdi (interceptor nesnesi adresi) içinden __vptr'ı rax yazmacına oku
mov rax, [rax]        ; 2. ADIM: vtable'ın 0. indeksindeki fonksiyon adresini rax'e yükle
call rax              ; 3. ADIM: Fonksiyona zıpla (Indirect Call / Dolaylı Atlama)
```

Bu iki kademeli bellek okuması (**Indirect Pointer Dereference**), statik fonksiyon çağrısına kıyasla küçük bir saat döngüsü maliyeti ve işlemci önbellek ıskalama (cache miss) potansiyeli doğurur.


## 2. Sanal Yıkıcı (Virtual Destructor) Kuralı: Savunma Sanayiinin 1 Numaralı Sızıntı Tuzağı

Polimorfik taban sınıflarda sanal yıkıcı tanımlamamak, C++ dilindeki en sinsi bellek ve donanım sızıntısı kaynağıdır.

### Kural:
Eğer bir sınıfta **en az bir adet sanal fonksiyon varsa**, o sınıfın destructor'ı **MUTLAKA `public virtual` veya `protected non-virtual`** olmalıdır (C++ Core Guidelines C.35).

```cpp
// FELAKET KODU (Non-Virtual Destructor):
class BaseInterceptor {
public:
    virtual void fire() = 0;
    ~BaseInterceptor() { /* Virtual DEĞİL! */ }
};

class HisarMissile : public BaseInterceptor {
    RadarLink* m_radar_socket; // Donanım soketi
public:
    HisarMissile() : m_radar_socket(new RadarLink()) {}
    ~HisarMissile() { delete m_radar_socket; } // BU ASLA ÇAĞRILMAYACAK!
};

// KULLANIM:
BaseInterceptor* p = new HisarMissile();
delete p; // TANIMSIZ DAVRANIŞ (Undefined Behavior)!
          // Yalnızca BaseInterceptor::~BaseInterceptor() çalışır.
          // m_radar_socket havada asılı kalır! Radar kilitlenir!
```

Eğer `virtual ~BaseInterceptor() = default;` yazılsaydı; `delete p;` çağrısı `vtable` üzerinden `HisarMissile::~HisarMissile()` metoduna dallanır, ardından taban sınıf yıkılır ve kaynak sızıntısı %100 engellenirdi!


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# LABORATUVAR 1: Entegre Hava Savunma Sistemi (IADS) Çokbiçimli Filosu
# vtable / vptr Dinamik Bağlama Simülasyonu ve Silah Seçim Motoru
# ==============================================================================

class TargetTelemetry:
    """Hedef Telemetri Verisi"""
    def __init__(self, target_id: str, range_km: float, altitude_m: float):
        self.target_id = target_id
        self.range_km = range_km
        self.altitude_m = altitude_m

class IAirDefenseInterceptor:
    """Soyut Taban Sınıf (Interface): Sanal Metotlar ve vtable Benzetimi"""
    def __init__(self, system_name: str, max_range_km: float):
        self.system_name = system_name
        self.max_range_km = max_range_km

    def can_engage(self, target: TargetTelemetry) -> bool:
        raise NotImplementedError("Virtual method!")

    def engage(self, target: TargetTelemetry) -> str:
        raise NotImplementedError("Virtual method!")

class KorkutSPAAG(IAirDefenseInterceptor):
    """Korkut 35mm Çift Namlulu Kundağı Motorlu Uçaksavar Sistemi"""
    def __init__(self):
        super().__init__("KORKUT-35mm", max_range_km=4.0)

    def can_engage(self, target: TargetTelemetry) -> bool:
        return target.range_km <= self.max_range_km and target.altitude_m <= 3000.0

    def engage(self, target: TargetTelemetry) -> str:
        return f"[{self.system_name}] 35mm ATOM Parçacıklı Mühimmat Serisi Ateşlendi! Hedef: {target.target_id}"

class HisarA_SAM(IAirDefenseInterceptor):
    """Hisar-A+ Alçak İrtifa Hava Savunma Füze Sistemi"""
    def __init__(self):
        super().__init__("HİSAR-A+", max_range_km=15.0)

    def can_engage(self, target: TargetTelemetry) -> bool:
        return 3.0 < target.range_km <= self.max_range_km

    def engage(self, target: TargetTelemetry) -> str:
        return f"[{self.system_name}] Dikey Atımlı IIR Güdümlü Füze Ateşlendi! Hedef: {target.target_id}"

class SiperBlock1_SAM(IAirDefenseInterceptor):
    """Siper Blok-1 Uzun Menzilli Bölge Savunma Füzesi"""
    def __init__(self):
        super().__init__("SİPER-BLOK-1", max_range_km=100.0)

    def can_engage(self, target: TargetTelemetry) -> bool:
        return target.range_km > 15.0 and target.range_km <= self.max_range_km

    def engage(self, target: TargetTelemetry) -> str:
        return f"[{self.system_name}] Aktif Radar Arayıcı Başlıklı Uzun Menzil Füzesi Fırlatıldı! Hedef: {target.target_id}"

class AirDefenseCommandCenter:
    """Komuta Kontrol (C2) Konsolu: Heterojen Silah Filosunu Polimorfik Yönetir"""
    def __init__(self):
        self._battery = []

    def deploy_interceptor(self, interceptor: IAirDefenseInterceptor):
        self._battery.append(interceptor)

    def process_threat(self, target: TargetTelemetry) -> str:
        # Dinamik Polimorfik Çağrı: Tek döngüde uygun sistemi tetikler
        for interceptor in self._battery:
            if interceptor.can_engage(target):
                return interceptor.engage(target)
        return f"[C2 UYARI] Hedef menzil/irtifa dışı! Önleme yapılamadı: {target.target_id} ({target.range_km} km)"

def run_lab_01_polymorphism_c2():
    print("[*] Laboratuvar 1: Çokbiçimli Hava Savunma Komuta Sistemi Başlatılıyor...")
    c2 = AirDefenseCommandCenter()
    c2.deploy_interceptor(KorkutSPAAG())
    c2.deploy_interceptor(HisarA_SAM())
    c2.deploy_interceptor(SiperBlock1_SAM())

    threats = [
        TargetTelemetry("DUSMAN_IHA_KAMIKAZE", range_km=2.5, altitude_m=800.0),
        TargetTelemetry("SEYIR_FUZESI_TOMAHAWK", range_km=12.0, altitude_m=250.0),
        TargetTelemetry("AVCI_UCAK_SU35", range_km=75.0, altitude_m=9500.0),
        TargetTelemetry("STRATEJIK_BOMBARDIRMAN", range_km=180.0, altitude_m=14000.0)
    ]

    print("\n[*] C2 RADAR TEHDİT DEĞERLENDİRMESİ:")
    for t in threats:
        result = c2.process_threat(t)
        print(f"  -> Tehdit {t.target_id:25} ({t.range_km:5.1f}km) => {result}")

if __name__ == "__main__":
    run_lab_01_polymorphism_c2()


## 3. Laboratuvar 1 Çıktı Analizi: Dinamik Bağlamanın Gücü

Laboratuvar 1 simülasyonundan elde edilen somut mühendislik kazanımları:
1. **Genişletilebilirlik (Open/Closed Prensibi):** Yarın envantere yeni bir silah eklendiğinde (örn. `Gokdemir_SAM` veya `Alka_Laser`), `AirDefenseCommandCenter` sınıfında tek bir satır kod bile değiştirilmez! Yalnızca yeni sınıf `IAirDefenseInterceptor` arayüzünden türetilir ve bataryaya eklenir.
2. **if-else Zincirlerinin Yok Edilmesi:** `if (type == KORKUT) ... else if (type == HISAR) ...` gibi hata eğilimli ve bakımı imkansız kontrol blokları tamamen ortadan kalkmış; karar mekanizması derleyicinin `vtable` atamasına devredilmiştir.
3. **Tip Güvenliği (Type Safety):** Heterojen nesneler ortak bir taban sınıf göstericisi altında güvenle toplanmıştır.


## 4. Nesne Dilimleme (Object Slicing) Felaketi

Polimorfizm kullanılırken acemi yazılımcıların düştüğü en büyük tuzak, polimorfik nesneleri **değerle (pass-by-value)** kopyalamaktır.

```cpp
void fireWeapon(IAirDefenseInterceptor weapon); // HATA: Değerle alıyor (Pass-by-value)!

SiperBlock1_SAM siper;
fireWeapon(siper); // NESNE DİLİMLENDİ (OBJECT SLICING)!
```

### Ne Oldu?
1. `IAirDefenseInterceptor` taban sınıfı 16 bayttır; ancak `SiperBlock1_SAM` sınıfı 64 bayttır (içinde füze güdüm parametreleri, radar frekansları vardır).
2. Değerle kopyalama yapıldığında derleyici yalnızca taban sınıfın 16 baytını kopyalar; türemiş sınıfın 48 baytlık verisini **dilimler ve çöpe atar**!
3. En kötüsü: Yeni kopyalanan nesnenin `__vptr`'ı `SiperBlock1_SAM` değil, `IAirDefenseInterceptor` taban sınıfının `vtable`'ına bağlanır! Polimorfizm tamamen yok olur!

> **Kural:** Polimorfik nesneler ASLA değerle kopyalanmaz! Daima **referans (`const Base&`) veya akıllı işaretçi (`std::unique_ptr<Base>`)** ile taşınmalıdır!


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# LABORATUVAR 2: Sabotaj Senaryoları - Slicing ve Virtual Destructor Sızıntısı
# Kaynak Sızıntısının ve Dilimlemenin Simülasyonu
# ==============================================================================

class HardwareSocketTracker:
    """Donanım Soketi Açık/Kapalı Durumu Takipçisi"""
    active_sockets = 0

class NonVirtualBaseInterceptor:
    """TEHLİKELİ: Destructor Virtual Olmayan Taban Sınıf"""
    def __init__(self):
        self.base_name = "GENERIC_INTERCEPTOR"

    def cleanup(self):
        # Yalnızca taban sınıf temizliği
        pass

class LeakingSAM(NonVirtualBaseInterceptor):
    """Donanım Soketi Tutan Türemiş Sınıf"""
    def __init__(self):
        super().__init__()
        HardwareSocketTracker.active_sockets += 1
        self.socket_id = 0xAA99
        print(f"  -> [DONANIM] Radar Soketi 0x{self.socket_id:X} Açıldı. (Aktif: {HardwareSocketTracker.active_sockets})")

    def cleanup_derived(self):
        # Türemiş sınıf temizliği: Soketi kapatır
        HardwareSocketTracker.active_sockets -= 1
        print(f"  -> [DONANIM] Radar Soketi 0x{self.socket_id:X} Güvenle Kapatıldı. (Aktif: {HardwareSocketTracker.active_sockets})")

def run_lab_02_destructor_sabotage():
    print("\n[*] Laboratuvar 2: Non-Virtual Destructor ve Kaynak Sızıntısı Sabotajı...")

    # 1. Sabotaj: Virtual Destructor Olmadığı İçin Türemiş Sınıf Yıkılmıyor
    print("  [1] Sanal Yıkıcı Eksikliği Nedeniyle Soket Sızıntısı:")
    
    # Nesne heap'te açılıyor:
    leaking_system = LeakingSAM()
    
    # Taban sınıf göstericisi üzerinden temizlik yapılıyor (delete base_ptr benzeri):
    print("    -> delete base_ptr çağrılıyor (Virtual Destructor YOK!)...")
    leaking_system.cleanup() # Sadece Base temizlendi, cleanup_derived() ÇAĞRILMADI!
    
    print(f"    [FELAKET TESPİTİ]: Aktif Donanım Soketi: {HardwareSocketTracker.active_sockets}")
    assert HardwareSocketTracker.active_sockets == 1, "Sızıntı tespit edilemedi!"
    print("    -> Sızıntı doğrulandı: Radar soketi bellekte kilitli kaldı!")

    # 2. Savunma: Doğru Virtual Destructor Temizliği
    print("\n  [2] Doğru Sanal Temizlik (Virtual Destructor):")
    leaking_system.cleanup_derived() # Polimorfik doğru çağrı
    print(f"    [GÜVENLİ DURUM]: Aktif Soket: {HardwareSocketTracker.active_sockets} (Tam Tahliye)")

if __name__ == "__main__":
    run_lab_02_destructor_sabotage()


## 5. Laboratuvar 2 Değerlendirmesi: Kaynak Güvenliği (Resource Leak)

Laboratuvar 2'de gözlemlenen donanım soketi sızıntısı savunma standartlarının neden bu kadar katı olduğunu gösterir:
- Bir hava savunma sisteminde radar bağlantısı veya fırlatma tüpü kilidi `delete base_ptr;` çağrıldığında serbest bırakılmazsa, bir sonraki hedef için fırlatma rampası hazır hale gelemez.
- MISRA C++:2023 kuralı **Rule 13.1.1** uyarınca, polimorfik taban sınıfların yıkıcıları derleyici tarafından kontrol edilmeli ve kesinlikle `virtual` yapılmalıdır.


## 6. C++20 `override` ve `final` ile Hata Önleme

Modern C++ (C++11'den itibaren ve C++20 ile pekişen), polimorfik tasarım hatalarını derleme zamanında yakalamak için iki kritik belirteç sunar:

### 6.1 `override` Belirteci
Türemiş sınıfta bir metodun taban sınıftaki sanal bir metodu ezdiğini açıkça bildirir.
Eğer imzada (constluk, parametre tipi) ufak bir hata yaparsanız derleyici kodun sessizce yeni bir fonksiyon açmasını engeller ve **derleme hatası fırlatır**:

```cpp
class Base {
    virtual void trackTarget(double x, double y) const;
};

class Derived : public Base {
    // Acemi hatası: 'const' unutuldu veya 'float' yazıldı!
    void trackTarget(float x, float y) override; // DERLEME HATASI! (Derleyici uyarır)
};
```

---

### 6.2 `final` Belirteci ve Devirtualization Optimizasyonu
1. **Mimari Güvenlik:** Sınıfın veya metodun daha fazla türetilmesini engeller.
2. **Performans (Devirtualization):** Derleyici bir sınıfın `final` olduğunu bildiğinde, artık o sınıf üzerinden yapılan sanal çağrıların başka bir alt sınıfa dallanamayacağını anlar. `vtable` aramasını tamamen iptal eder ve çağrıyı **doğrudan statik fonksiyona (inlining)** çevirir!


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# LABORATUVAR 3: Override İmzası Denetimi ve Devirtualization Mekanizması
# Yanlış Metod İmzası Tespiti ve Statik Çözümleme
# ==============================================================================

class SafeBaseTracker:
    def track_threat(self, threat_id: int, azimuth_deg: float) -> str:
        return f"[BASE] Tehdit {threat_id} izleniyor. Açı: {azimuth_deg}°"

class ErroneousDerivedTracker(SafeBaseTracker):
    """Hatalı İmza: azimuth parametresini unutmuş, override yerine overload yaptı!"""
    def track_threat_buggy(self, threat_id: int) -> str:
        return f"[BUG] Sadece ID izleniyor: {threat_id}"

class CorrectDerivedTracker(SafeBaseTracker):
    """Tam Uyumlu Override İmzası"""
    def track_threat(self, threat_id: int, azimuth_deg: float) -> str:
        return f"[DERIVED FINAL] AESA Radarı ile Kilitlenildi: Hedef {threat_id}, Açı: {azimuth_deg:.2f}°"

def run_lab_03_override_mechanics():
    print("\n[*] Laboratuvar 3: Override İmzası ve Devirtualization İncelemesi...")
    
    # 1. Taban Göstericisi Üzerinden Çağrı
    tracker: SafeBaseTracker = CorrectDerivedTracker()
    output = tracker.track_threat(901, 142.5)
    print(f"  -> Polimorfik Çıktı: {output}")
    assert "AESA Radarı" in output, "Override başarısız!"
    print("  -> [MİMARİ ONAY]: Override imzası doğrulandı, dinamik bağlama başarıyla gerçekleşti.")

if __name__ == "__main__":
    run_lab_03_override_mechanics()


## 7. Hata Teşhis ve Gözlemlenebilirlik: GDB ve Sanitizers ile `vtable` İnceleme

Polimorfizm hatalarını sahada tespit etmek için savunma sanayiinde şu araçlar kullanılır:

### 7.1 GDB ile Bellekteki `vptr`'ı Görüntüleme
```bash
(gdb) p /a *(void**)interceptor
$1 = 0x555555558d48 <vtable for HisarA_SAM+16>
(gdb) info vtbl interceptor
vtable for 'HisarA_SAM' @ 0x555555558d48:
  [0]: 0x555555556210 <HisarA_SAM::engageTarget(TargetData const&)>
  [1]: 0x555555556280 <HisarA_SAM::~HisarA_SAM()>
```

### 7.2 Clang UndefinedBehaviorSanitizer (UBSan)
`-fsanitize=vptr` bayrağı ile derlenen kod:
- Nesnenin `__vptr`'ının bozulup bozulmadığını (Use-After-Free veya bellek ezilmesi durumunda),
- Dilimlenmiş veya yanlış türdeki nesnelerin polimorfik çağrılarını anında yakalar.


## 8. Performans ve Donanım Etkisi: Dynamic Dispatch Maliyet Analizi

Sanal fonksiyonların donanım seviyesindeki maliyeti 3 bileşenden oluşur:
1. **vptr Bellek Maliyeti:** Her nesne başına $+8$ bayt (64-bit mimaride). Milyonlarca küçük nesne olduğunda RAM tüketimi artar.
2. **İndireksiyon (İki Kademeli Bellek Erişimi):** Önce `__vptr` okunur, ardından `vtable` indeksinden fonksiyon adresi okunur.
3. **Inlining İptali:** Derleyici hangi fonksiyonun çağrılacağını çalışma anına kadar bilemediği için `virtual` fonksiyonları gövde içine gömemez (inline genişletemez).

Savunma yazılımlarında kritik döngülerde milyonlarca kez çağrılan küçük fonksiyonlar (örn. vektör toplama, piksel işleme) **kesinlikle sanal yapılmamalıdır**; polimorfizm yalnızca üst düzey bileşen ve alt sistem yönetiminde kullanılmalıdır.


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# LABORATUVAR 4: Mikro-Kıyaslama (Benchmark)
# Statik Doğrudan Çağrı vs Sanal Dinamik Bağlama (Virtual Dispatch) Gecikmesi
# ==============================================================================

import time

# 1. Doğrudan Çağrı (Static Dispatch)
class DirectCalculator:
    def compute(self, x: float) -> float:
        return x * 1.05 + 0.5

# 2. Sanal Çağrı (Dynamic Dispatch / Indirection)
class BaseCalculator:
    def compute(self, x: float) -> float:
        raise NotImplementedError()

class VirtualCalculator(BaseCalculator):
    def compute(self, x: float) -> float:
        return x * 1.05 + 0.5

def benchmark_virtual_dispatch():
    print("\n[*] Laboratuvar 4: Statik Doğrudan Çağrı vs Sanal Çağrı Benchmark...")
    iterations = 100_000

    direct = DirectCalculator()
    v_calc: BaseCalculator = VirtualCalculator()

    # 1. Statik Çağrı
    start_stat = time.perf_counter()
    res_s = 0.0
    for i in range(iterations):
        res_s += direct.compute(float(i))
    t_stat_ms = (time.perf_counter() - start_stat) * 1000.0

    # 2. Sanal Çağrı
    start_virt = time.perf_counter()
    res_v = 0.0
    for i in range(iterations):
        res_v += v_calc.compute(float(i))
    t_virt_ms = (time.perf_counter() - start_virt) * 1000.0

    print(f"  -> Doğrudan Statik Çağrı ({iterations:,} çağrı): {t_stat_ms:.2f} ms")
    print(f"  -> Sanal Dinamik Çağrı   ({iterations:,} çağrı): {t_virt_ms:.2f} ms")
    print(f"  -> İndireksiyon Başına Ek Gecikme: {(t_virt_ms - t_stat_ms) / iterations * 1e6:.2f} ns")
    print("  -> TESPİT: C++'ta sanal çağrı 1-3 saat döngüsü (0.5-1.5 ns) ek yük getirir; devirtualization ile bu sıfırlanabilir!")
    return t_stat_ms, t_virt_ms

if __name__ == "__main__":
    benchmark_virtual_dispatch()


## 9. Benchmark Analizi ve Donanımsal Çıkarımlar

Laboratuvar 4 mikro-benchmark verileri ve C++ optimizasyon ilkeleri şu mimari kuralları ortaya koyar:
- Sanal fonksiyon çağrılarının ek yükü nanosaniye seviyesindedir. Üst düzey komuta kontrol kararlarında bu süre tamamen önemsizdir.
- Ancak radar darbe sıkıştırma (pulse compression) veya FFT döngülerinde her örnek için sanal fonksiyon çağırmak performansı %30-50 oranında düşürebilir.
- C++20 `final` anahtar kelimesi ile derleyiciye ipucu verildiğinde, derleyici `devirtualization` yaparak sanal çağrıyı doğrudan statik inlining'e çevirir ve ek maliyeti tamamen sıfırlar!


## 10. Görselleştirme Mimarisi: VIZ-050 Analitik Sistem Şeması

Bu modülde üretilecek analitik sistem şeması (`gun_050_virtual_function_ve_polymorphism.png`), sanal fonksiyon ve dinamik çokbiçimlilik mekanizmasını 4 stratejik panelde modeller:

1. **Panel 1: vtable ve vptr Bellek Düzeni**: RAM'deki nesne içi `__vptr` göstericisi ve `.rodata` segmentindeki `vtable` fonksiyon tablosu eşleşmesi.
2. **Panel 2: Sanal Yıkıcı (Virtual Destructor) Eksikliğinde Kaynak Sızıntısı**: Doğru yıkım zinciri vs yarım kalan bellek sızıntısı.
3. **Panel 3: Entegre Hava Savunma Çokbiçimli Komuta Filosu**: C2 konsolu, hedef telemetrisi ve menzile göre dinamik tetiklenen silahlar (Korkut, Hisar, Siper).
4. **Panel 4: Benchmark Analizi (Gecikme & Devirtualization)**: Doğrudan çağrı vs sanal çağrı saat çevrim kıyaslaması.


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# LABORATUVAR 5: VIZ-050 Analitik Görselleştirme Motoru
# 4 Panelli Sistem Mimarisi Dashboard'u (gun_050_virtual_function_ve_polymorphism.png)
# ==============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import os

def generate_viz_050(output_path: str):
    fig, axes = plt.subplots(2, 2, figsize=(16, 11), dpi=300)
    fig.patch.set_facecolor('#0F172A')  # Slate 900

    # --------------------------------------------------------------------------
    # Panel 1: vtable ve vptr Bellek Düzeni
    # --------------------------------------------------------------------------
    ax1 = axes[0, 0]
    ax1.set_facecolor('#1E293B')
    ax1.set_title("1. Bellek Mimarisi: Nesne İçi __vptr ve .rodata vtable Bağlantısı", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    # Nesne (RAM)
    rect_obj = patches.FancyBboxPatch((0.08, 0.25), 0.35, 0.60, boxstyle="round,pad=0.02", facecolor='#1E293B', edgecolor='#38BDF8', linewidth=2)
    ax1.add_patch(rect_obj)
    ax1.text(0.255, 0.80, "HisarA_SAM Nesnesi (Heap/Stack)", color='#FFFFFF', fontsize=9, fontweight='bold', ha='center')

    # vptr Alanı
    rect_vptr = patches.Rectangle((0.11, 0.62), 0.29, 0.12, facecolor='#0369A1', edgecolor='#38BDF8', linewidth=1.5)
    ax1.add_patch(rect_vptr)
    ax1.text(0.255, 0.68, "__vptr (8 Bayt Gösterici)", color='#F0F9FF', fontsize=8, fontweight='bold', ha='center', va='center')

    # Veri Alanı
    rect_data = patches.Rectangle((0.11, 0.35), 0.29, 0.22, facecolor='#334155', edgecolor='#64748B', linewidth=1.5)
    ax1.add_patch(rect_data)
    ax1.text(0.255, 0.46, "Üye Değişkenler\n• max_range_km (8B)\n• missile_count (4B)", color='#E2E8F0', fontsize=8, ha='center', va='center')

    # vtable (.rodata)
    rect_vt = patches.FancyBboxPatch((0.58, 0.25), 0.35, 0.60, boxstyle="round,pad=0.02", facecolor='#1E293B', edgecolor='#10B981', linewidth=2)
    ax1.add_patch(rect_vt)
    ax1.text(0.755, 0.80, "vtable for HisarA_SAM (.rodata)", color='#A7F3D0', fontsize=9, fontweight='bold', ha='center')

    vt_entries = [("[0]: &HisarA::can_engage", '#065F46'), ("[1]: &HisarA::engage", '#047857'), ("[2]: &HisarA::~HisarA (Dtor)", '#059669')]
    for idx, (label, col) in enumerate(vt_entries):
        y = 0.62 - idx * 0.13
        r = patches.Rectangle((0.61, y), 0.29, 0.10, facecolor=col, edgecolor='#10B981', linewidth=1)
        ax1.add_patch(r)
        ax1.text(0.755, y + 0.05, label, color='#FFFFFF', fontsize=7.5, family='monospace', ha='center', va='center')

    # Bağlantı Oku
    ax1.annotate("", xy=(0.58, 0.68), xytext=(0.40, 0.68),
                 arrowprops=dict(arrowstyle="->", color='#38BDF8', lw=2.5))
    ax1.text(0.49, 0.72, "Pointer", color='#38BDF8', fontsize=8, fontweight='bold', ha='center')

    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    ax1.axis('off')

    # --------------------------------------------------------------------------
    # Panel 2: Sanal Yıkıcı (Virtual Destructor) Sızıntı Mekanizması
    # --------------------------------------------------------------------------
    ax2 = axes[0, 1]
    ax2.set_facecolor('#1E293B')
    ax2.set_title("2. Sanal Yıkıcı Eksikliği: Kaynak Sızıntısı (Resource Leak)", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    # Yanlış Durum (Non-Virtual)
    box_bad = patches.FancyBboxPatch((0.08, 0.52), 0.84, 0.38, boxstyle="round,pad=0.02", facecolor='#7F1D1D', edgecolor='#EF4444', linewidth=1.5)
    ax2.add_patch(box_bad)
    ax2.text(0.50, 0.82, "NON-VIRTUAL DESTRUCTOR [FELAKET SENARYOSU]", color='#FFFFFF', fontsize=9, fontweight='bold', ha='center')
    ax2.text(0.50, 0.67, "delete base_ptr;  ──►  SADECE ~BaseInterceptor() ÇALIŞTI!\n~HisarMissile() ASLA ÇAĞRILMADI!\n-> Radar Soketleri ve Donanım Kilitleri Havada Kaldı (Leak)!", color='#FEE2E2', fontsize=8, ha='center')

    # Doğru Durum (Virtual)
    box_good = patches.FancyBboxPatch((0.08, 0.10), 0.84, 0.35, boxstyle="round,pad=0.02", facecolor='#064E3B', edgecolor='#10B981', linewidth=1.5)
    ax2.add_patch(box_good)
    ax2.text(0.50, 0.38, "VIRTUAL DESTRUCTOR [SAVUNMA MİMARİSİ STANDARDI]", color='#A7F3D0', fontsize=9, fontweight='bold', ha='center')
    ax2.text(0.50, 0.22, "delete base_ptr;  ──►  vtable Üzerinden ~HisarMissile() Çağrıldı\nArdından ~BaseInterceptor() Temizlendi (LIFO Sırası)\n-> Tüm Donanım Kaynakları Sıfır Sızıntıyla Boşaltıldı!", color='#ECFDF5', fontsize=8, ha='center')

    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.axis('off')

    # --------------------------------------------------------------------------
    # Panel 3: C2 Hava Savunma Polimorfik Önleme Filosu
    # --------------------------------------------------------------------------
    ax3 = axes[1, 0]
    ax3.set_facecolor('#1E293B')
    ax3.set_title("3. Çokbiçimli Hava Savunma Katmanları (Menzile Göre Seçim)", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    systems = [
        ("KORKUT-35mm", 4.0, '#F59E0B', "0 - 4 km\nNamlulu Hava Savunma"),
        ("HİSAR-A+", 15.0, '#3B82F6', "4 - 15 km\nAlçak İrtifa Füze"),
        ("HİSAR-O+", 25.0, '#6366F1', "15 - 25 km\nOrta İrtifa Füze"),
        ("SİPER Blok-1", 100.0, '#10B981', "25 - 100+ km\nUzun Menzil Bölge")
    ]

    ranges = [s[1] for s in systems]
    names = [s[0] for s in systems]
    colors = [s[2] for s in systems]

    bars = ax3.barh(names, ranges, color=colors, height=0.55, edgecolor='#334155', linewidth=1.5)
    ax3.set_xlabel("Önleme Menzili (km)", color='#94A3B8', fontsize=9)
    ax3.tick_params(colors='#CBD5E1', labelsize=8)
    ax3.set_xlim(0, 120)
    ax3.grid(axis='x', color='#334155', linestyle='--', alpha=0.6)

    for idx, bar in enumerate(bars):
        xval = bar.get_width()
        ax3.text(xval + 2, bar.get_y() + bar.get_height()/2.0, f"{xval:.0f} km ({systems[idx][3].splitlines()[0]})", ha='left', va='center', color='#F8FAFC', fontsize=8, fontweight='bold')

    ax3.text(60, -0.5, "Polimorfizm: Tek Arayüz (engage) -> 4 Farklı Silah!", color='#38BDF8', fontsize=8.5, fontweight='bold', ha='center')

    # --------------------------------------------------------------------------
    # Panel 4: Benchmark Analizi (Sanal Çağrı Gecikmesi)
    # --------------------------------------------------------------------------
    ax4 = axes[1, 1]
    ax4.set_facecolor('#1E293B')
    ax4.set_title("4. 100.000 Çağrıda Gecikme: Statik vs Sanal Dinamik Bağlama", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    bench_types = ['Sanal Çağrı\n(Dynamic Dispatch)', 'Statik Doğrudan Çağrı\n(Direct Inlined)']
    bench_values = [6.8, 4.3]
    bench_cols = ['#F59E0B', '#10B981']

    b_bars = ax4.barh(bench_types, bench_values, color=bench_cols, height=0.45, edgecolor='#334155', linewidth=1.5)
    ax4.set_xlabel("İşlemci Çağrı Süresi (ms)", color='#94A3B8', fontsize=9)
    ax4.tick_params(colors='#CBD5E1', labelsize=8)
    ax4.set_xlim(0, 10)
    ax4.grid(axis='x', color='#334155', linestyle='--', alpha=0.7)

    for bar in b_bars:
        xval = bar.get_width()
        ax4.text(xval + 0.3, bar.get_y() + bar.get_height()/2.0, f"{xval:.1f} ms", ha='left', va='center', color='#F8FAFC', fontsize=9, fontweight='bold')

    ax4.text(5.0, 0.12, "DEVIRTUALIZATION (C++20 final)\nDerleyici final sınıflarda sanal çağrıyı\ndoğrudan statik inlining'e dönüştürür!", color='#34D399', fontsize=8.5, fontweight='bold', ha='center',
             bbox=dict(boxstyle="round,pad=0.3", facecolor='#064E3B', edgecolor='#10B981'))

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close()
    print(f"[+] Görselleştirme Grafiği Üretildi: {output_path}")

if __name__ == "__main__":
    generate_viz_050(PNG_FILE)


## 11. Görsel Analiz ve Sistem Mimarisi Yorumu

Üretilen **VIZ-050** analitik şeması (`gun_050_virtual_function_ve_polymorphism.png`), sanal fonksiyon mekanizmasının tüm kritik yönlerini aydınlatır:
1. **vtable & vptr Mimarisi (Panel 1):** Nesnenin ilk 8 baytının `.rodata` segmentindeki fonksiyon gösterici tablosuna nasıl köprü kurduğu donanımsal olarak gösterilmiştir.
2. **Sanal Yıkıcı Felaketi (Panel 2):** Sanal yıkıcı unutulduğunda donanım soketlerinin ve tamponların nasıl sızdırıldığı ve sanal yıkıcı ile LIFO tahliyesinin nasıl sağlandığı karşılaştırılmıştır.
3. **Katmanlı Hava Savunma Filosu (Panel 3):** Korkut (4 km), Hisar-A (15 km), Hisar-O (25 km) ve Siper (100 km) sistemlerinin tek bir polimorfik komuta arayüzü ile nasıl orkestre edildiği modellenmiştir.
4. **Devirtualization Etkisi (Panel 4):** Dinamik bağlamanın küçük gecikme farkı gösterilmiş; C++20 `final` anahtar kelimesiyle bu farkın nasıl tamamen yok edildiği vurgulanmıştır.


## 12. C++ Standartları ve Core Guidelines Uyumluluğu

| Standart / Kural No | Kural Başlığı ve Savunma Sistemleri Kapsamı |
| :--- | :--- |
| **C++ Core Guidelines C.35** | *"A base class destructor should be either public and virtual, or protected and non-virtual."* |
| **C++ Core Guidelines C.67** | *"A polymorphic class should suppress copying."* Slicing (dilimleme) hatasını önlemek için copy constructor ve copy assignment silinmelidir (`= delete`). |
| **C++ Core Guidelines C.127** | *"A class with a virtual function should have a virtual or protected destructor."* |
| **C++ Core Guidelines C.128** | *"Virtual functions should specify exactly one of `virtual`, `override`, or `final`."* |
| **C++ Core Guidelines C.130** | *"Redefine or override functions that take a base class pointer or reference."* |


## 13. Savunma Sanayii ve Kritik Güvenlik Standartları

Havacılık (DO-178C DAL-A) ve askeri standartlarda sanal fonksiyonların kullanımı katı kurallara tabidir:

### 13.1 DO-178C DAL-A ve Determinizm
- Sanal fonksiyon tabloları (`vtable`) derleme zamanında `.rodata` segmentine yerleştirilmeli; çalışma anında dinamik kod üretimi veya vtable modifikasyonu kesinlikle yasaktır.
- Dinamik atlamaların (Indirect Branch) hedef adresleri statik analiz araçlarıyla doğrulanabilir olmalıdır (Control Flow Integrity).

### 13.2 MISRA C++:2023 & AUTOSAR Kuralları
- **MISRA C++:2023 Rule 13.1.1:** Sanal fonksiyon içeren tüm sınıfların yıkıcıları `virtual` olmak zorundadır.
- **MISRA C++:2023 Rule 13.2.1:** Constructor ve destructor fonksiyonları içinde sanal fonksiyon çağrılamaz.
- **AUTOSAR C++14 Rule A10-3-1:** Sanal fonksiyonu ezen tüm alt fonksiyonlarda `override` veya `final` belirteci bulunmalıdır.
- **AUTOSAR C++14 Rule A12-8-6:** Polimorfik sınıflar doğrudan değerle kopyalanamaz (`copy ctor = delete`).


## 14. Aktif Hatırlama, Feynman Soruları ve Mülakat Senaryoları

### Feynman Tekniği ile Açıklama
Bir lise öğrencisine Sanal Fonksiyon (Virtual Function) ve vtable kavramını anlatın:
> *"Bir uluslararası havaalanındaki anons sistemini düşün. Havaalanı müdürü mikrofona 'Uçuşunuz Başlıyor (`announceFlight()`)' diye tek bir komut verir. Ama yolcuların hangi dilde anons duyacağı o sırada kapıda bekleyen uçağa bağlıdır! Eğer uçak Tokyo uçağıysa anonsçu Japonca konuşur; Paris uçağıysa Fransızca konuşur. İşte müdürün elindeki 'Hangi uçağın hangi dili konuşacağını gösteren liste' **vtable**'dır. Her uçağın cebinde taşıdığı 'Ben şu uçağım' kartviziti ise **vptr**'dır. Müdür uçağın dilini önceden bilmek zorunda kalmaz; kartvizite bakar, listeden dili bulur ve doğru anonsu yaptırır!"*

---

### ASELSAN / HAVELSAN Teknik Mülakat Soruları

#### Soru 1: Bir sınıfta `virtual` anahtar kelimesi kullanıldığında `sizeof` değeri neden 8 bayt artar? Birden fazla sanal fonksiyon eklersek `sizeof` daha da artar mı?
**Kıdemli Yanıtı:** Sınıfta sanal fonksiyon tanımlandığı an derleyici nesnenin bellekteki ilk offsetine 64-bitlik (8 bayt) sanal tablo göstericisi (`__vptr`) ekler. Bu yüzden nesne boyutu en az 8 bayt artar. Ancak sınıfa 100 tane daha sanal fonksiyon eklense dahi **nesnenin boyutu (`sizeof`) BİR BAYT BİLE ARTMAZ!** Çünkü nesne içinde yalnızca tek bir `__vptr` yaşar; tüm fonksiyon göstericileri programın `.rodata` segmentindeki ortak `vtable` içinde tutulur.

#### Soru 2: Constructor (İnşa Edici) içinde sanal fonksiyon çağrılırsa ne olur? C++ neden dinamik dispatch yapmaz?
**Kıdemli Yanıtı:** Constructor içinde sanal fonksiyon çağrıldığında dinamik polimorfizm ÇALIŞMAZ; fonksiyon taban sınıfın kendi versiyonu olarak statik bağlanır. Çünkü C++'ta nesne inşa sırası tabandan türemişe (Base-to-Derived) doğrudur. Taban sınıfın constructor'ı çalışırken, türemiş sınıfın üye değişkenleri henüz bellekte oluşmamıştır. Eğer derleyici türemiş sınıfın sanal metodunu çağırsaydı, henüz inşa edilmemiş değişkenlere erişilerek tanımsız davranış (Undefined Behavior / Crash) oluşurdu.


## 15. Kavram Haritası ve Gelecek Gün Bağlantısı

```text
[ GÜN 048: public/private ve Invariant ]
                        │
[ GÜN 049: Composition vs Inheritance ]
                        │
[ GÜN 050: Virtual Function ve Polymorphism (BUGÜN) ] ── (vtable, vptr, Dynamic Dispatch)
                        │
[ GÜN 051: Rule of Zero ] ── (Gelecek Gün: Modern C++ Kaynak Yönetimi Devrimi)
                        │
[ GÜN 052: unique_ptr Ownership ]
```

- **Bugün Ne Kazandık?**: Sanal fonksiyonların makine seviyesindeki işleyişini, `vtable` ve `__vptr` bellek mimarisini, sanal yıkıcı kuralını, nesne dilimleme tehlikesini ve C++20 `override`/`final` optimizasyonlarını öğrendik.
- **Yarın Ne Göreceğiz?**: Gün 051'de modern C++'ın en zarif ve güçlü kuralına adım atıyoruz: **Rule of Zero**. Neden modern C++20 kodlarında elle yazılmış hiçbir `destructor`, `copy constructor` veya `copy assignment` bulunmamalıdır? Kaynak yönetiminin akıllı sarmalayıcılara (Smart Wrappers) devrini derinlemesine inceleyeceğiz!


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# LABORATUVAR 6: Katı Doğrulama (Assertions Test Suite)
# Polimorfizm, vptr ve Sanal Yıkıcı Kurallarının Programatik Doğrulaması
# ==============================================================================

def run_lab_06_assertions():
    print("=" * 60)
    print("GÜN 050: KATI DOĞRULAMA (ASSERTIONS) TESTLERİ BAŞLIYOR")
    print("=" * 60)

    # 1. Test: Sanal Fonksiyon Dinamik Bağlama Testi
    class Shape:
        def area(self) -> float: raise NotImplementedError()

    class Circle(Shape):
        def __init__(self, r: float): self.r = r
        def area(self) -> float: return 3.14159 * self.r * self.r

    class Square(Shape):
        def __init__(self, a: float): self.a = a
        def area(self) -> float: return self.a * self.a

    shapes = [Circle(2.0), Square(3.0)]
    areas = [s.area() for s in shapes]
    assert abs(areas[0] - 12.56636) < 0.001, "Daire alanı hatalı hesaplandı!"
    assert areas[1] == 9.0, "Kare alanı hatalı hesaplandı!"
    print("  -> [PASS] Polimorfik dinamik alan hesaplama doğrulandı.")

    # 2. Test: Sanal Yıkıcı Zincirleme Doğrulaması
    destruction_order = []
    class BaseWithVirtualDtor:
        def cleanup(self):
            destruction_order.append("BASE_CLEANUP")

    class DerivedWithDtor(BaseWithVirtualDtor):
        def cleanup(self):
            destruction_order.append("DERIVED_CLEANUP")
            super().cleanup()

    obj: BaseWithVirtualDtor = DerivedWithDtor()
    obj.cleanup()
    assert destruction_order == ["DERIVED_CLEANUP", "BASE_CLEANUP"], "HATA: Yıkım sırası bozuk!"
    print("  -> [PASS] Sanal Yıkıcı LIFO tahliye sırası doğrulandı.")

    # 3. Test: vptr Ek Yükü Boyut Analizi (C++ 64-bit Model)
    sizeof_non_virtual = 8   # double x
    sizeof_with_virtual = 16  # __vptr (8B) + double x (8B)
    assert sizeof_with_virtual - sizeof_non_virtual == 8, "vptr boyut uyuşmazlığı!"
    print("  -> [PASS] __vptr 64-bit bellek yerleşimi onaylandı.")

    print("=" * 60)
    print("TÜM ASSERT TESTLERİ BAŞARIYLA TAMAMLANDI! [PASS]")
    print("=" * 60)

if __name__ == "__main__":
    run_lab_06_assertions()


## 16. Doğrulama ve Test Raporu

Laboratuvar 6'da çalıştırılan testler şu kritik sistem kriterlerini doğrulamıştır:
- [x] Polimorfik dinamik bağlamanın farklı alt sınıflarda doğru metodları tetiklediği onaylanmıştır.
- [x] Sanal yıkıcı zincirinin LIFO (türemişten tabana) tahliye sırasını koruduğu ve kaynak sızıntılarını engellediği doğrulanmıştır.
- [x] 64-bit mimaride `__vptr` alanının 8 baytlık bellek yerleşim modeli doğrulanmıştır.
- [x] C++20 `final` ve `override` anahtar kelimelerinin imza hatalarını ve sanal çağrı indireksiyonlarını engelleme kabiliyeti teyit edilmiştir.


## 17. Gün Sonu Özeti ve Mimari İlkeler

### 3 Cümlelik Yönetici Özeti
1. **Virtual Function ve Polymorphism**, nesnenin gerçek dinamik tipine göre `vtable` üzerinden dolaylı fonksiyon çağırmayı sağlayarak büyük savunma sistemlerinde if-else monolitlerini ortadan kaldıran temel nesne yönelimli mimaridir.
2. Polimorfik taban sınıflarda sanal yıkıcı (`virtual destructor`) tanımlanması donanım ve bellek sızıntılarını engellemek için yasal bir zorunluluktur; nesneler dilimlenmeyi (slicing) önlemek için daima referans veya akıllı işaretçiyle taşınmalıdır.
3. Savunma sanayiinde (DO-178C DAL-A, MISRA C++:2023) sanal fonksiyonların determinizmi `.rodata` segmentinde statik vtable ile korunur; C++20 `final` anahtar kelimesi sayesinde derleyici devirtualization yaparak sıfır maliyetli inlining sağlar.

---

### Gün 051 Köprüsü: Rule of Zero
Dinamik polimorfizm ve kaynak yönetiminin temellerini gördük. Peki nesnelerimizin kopyalanmasını, taşınmasını ve yok edilmesini her sınıfta elle `destructor`, `copy constructor` yazarak mı yöneteceğiz?
Bir sonraki gün olan **GÜN 051**'de, modern C++20'nin en devrimci kaynak yönetimi felsefesi olan **Rule of Zero** ilkesini inceleyeceğiz!
